# Step-1 : Install  the required libraries/packages

In [ ]:
# pip install pandas
# pip install openpyxl


# Step-2 : Import the required libraries/packages

In [2]:
import pandas as pd

# Step 3: Load/Import data & Build connection

In [3]:
df = pd.read_excel("customer_sample_500.csv.xlsx", dtype=str)

# Replace NaN with empty string
df = df.fillna("")

print("File loaded successfully")

File loaded successfully


# Step 4: Data Quality | Completeness Check

## Datasets using: 
- customer_sample_500


In [4]:
import pandas as pd
import os

#------Completeness check function-------

def completeness_check(df, output_file="QualityCheck_results.xlsx"):

    results = []

    # Loop through each column in the dataframe
    for col in df.columns:

        # Count empty or whitespace values
        missing = df[col].astype(str).str.strip().eq("").sum()

        results.append({
            "Column": col,
            "Missing Values": missing
        })

    # Convert results list into dataframe
    result_df = pd.DataFrame(results)

    # Results folder
    folder_path = "results"

    # Create folder if it doesn't exist
    os.makedirs(folder_path, exist_ok=True)

    # Full file path
    file_path = os.path.join(folder_path, output_file)

    # Write to Excel
    if os.path.exists(file_path):

        with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            result_df.to_excel(writer, sheet_name="Completeness_Check", index=False)

    else:

        with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
            result_df.to_excel(writer, sheet_name="Completeness_Check", index=False)

    print("Output saved to:", file_path)

    return result_df


# Run completeness check
result = completeness_check(df)

Output saved to: results\QualityCheck_results.xlsx


# Step -5: Data Quality | Uniquenss check
## Datasets using:
- customer_sample_500

In [5]:
import pandas as pd
import os

#--uniqueness_check-----

# ---- Step 1: Find columns that contain unique values ----
def find_unique_columns(df):

    unique_cols = []

    for col in df.columns:
        if df[col].nunique(dropna=False) == len(df):
            unique_cols.append(col)

    return unique_cols


# ---- Step 2: Perform uniqueness check ----
def uniqueness_check(df, output_file="QualityCheck_results.xlsx"):

    unique_cols = find_unique_columns(df)

    # DataFrame for unique columns
    unique_df = pd.DataFrame(unique_cols, columns=["Unique_Columns"])

    duplicate_results = []

    for col in unique_cols:

        duplicates = (df[col].value_counts() > 1).sum()

        duplicate_results.append({
            "Column": col,
            "Duplicate_Count": duplicates
        })

    duplicate_df = pd.DataFrame(duplicate_results)

    # Results folder
    folder_path = "results"
    os.makedirs(folder_path, exist_ok=True)

    file_path = os.path.join(folder_path, output_file)

    # Write both sheets
    if os.path.exists(file_path):

        with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            unique_df.to_excel(writer, sheet_name="Uniqueness_Columns", index=False)
            duplicate_df.to_excel(writer, sheet_name="Uniqueness_Check", index=False)

    else:

        with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
            unique_df.to_excel(writer, sheet_name="Uniqueness_Columns", index=False)
            duplicate_df.to_excel(writer, sheet_name="Uniqueness_Check", index=False)

    print("Output saved in:", file_path)

    return duplicate_df


# Run uniqueness check
result = uniqueness_check(df)

Output saved in: results\QualityCheck_results.xlsx


# Step-6 : Quality check | Validity check
## Datasets using :
- customer_sample_500 

In [6]:
def run_validity_checks(df, output_file="QualityCheck_results.xlsx"):

    validity_rules = {
        "Invalid_SIRET": (df["SIRET"].str.len() != 14),

        "Invalid_SIREN": (df["SIREN"].str.len() != 9),

        "Invalid_Country_Code": (df["COUNTRY_CD"].str.len() != 2),

        "Invalid_SAP_Postal_Code": (~df["SAP_POSTAL_ZIP_CD"].str.match(r'^\d+$')),

        "Invalid_LUCI_Postal_Code": (~df["LUCI_POSTAL_ZIP_CD"].str.match(r'^\d+$')),

        "Invalid_Customer_Delivery_Block":
            (~df["CUSTOMER_DELIVERY_BLOC"].isin(["00", "01"])),

        "Invalid_GERS":
            (~df["GERS"].str.match(r'^\d+$'))
    }

    delivery_columns = [
        "CUSTOMER_ORDER_BLOCK",
        "CUSTOMER_SALES_DELIVERY_BLOC",
        "CUSTOMER_SALES_ORDER_BLOCK",
        "DELETION_BLOCK"
    ]

    # Add rules dynamically
    for col in delivery_columns:
        validity_rules[f"{col}_Invalid"] = ~df[col].isin(["00", "01"])

    results = []

    for rule, condition in validity_rules.items():

        invalid_count = condition.sum()

        results.append({
            "Check": rule,
            "Invalid_Count": invalid_count
        })

    validity_df = pd.DataFrame(results)

    # Results folder
    folder_path = "results"
    os.makedirs(folder_path, exist_ok=True)

    file_path = os.path.join(folder_path, output_file)

    # Write sheet
    if os.path.exists(file_path):

        with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            validity_df.to_excel(writer, sheet_name="Validity_Check", index=False)

    else:

        with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
            validity_df.to_excel(writer, sheet_name="Validity_Check", index=False)

    print("Output saved in:", file_path)

    return validity_df


# Run validity checks
result = run_validity_checks(df)

Output saved in: results\QualityCheck_results.xlsx


# Step-7: Quality Checks | Consistency check
## Data sets using :
- customer_sample_500

In [ ]:
# Consistency check 

def run_consistency_checks(df, output_file="QualityCheck_results.xlsx"):

    # Dictionary defining consistency rules
    consistency_rules = {
        "Country_Mismatch": df["COUNTRY_CD"] != df["LUCI_COUNTRY_CD"]
    }

    results = []

    # Loop through rules and count mismatches
    for rule, condition in consistency_rules.items():

        mismatch_count = condition.sum()

        results.append({
            "Check": rule,
            "Mismatch_Count": mismatch_count
        })

    # Convert results to DataFrame
    consistency_df = pd.DataFrame(results)

    # Results folder
    folder_path = "results"
    os.makedirs(folder_path, exist_ok=True)

    file_path = os.path.join(folder_path, output_file)

    # Write to Excel
    if os.path.exists(file_path):

        with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            consistency_df.to_excel(writer, sheet_name="Consistency_Check", index=False)

    else:

        with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
            consistency_df.to_excel(writer, sheet_name="Consistency_Check", index=False)

    print("Output saved in:", file_path)

    return consistency_df


# Run consistency check
result = run_consistency_checks(df)

Output saved in: results\QualityCheck_results.xlsx
